## Bronze Ingest
Ingests raw CSV and JSON invoices into Bronze with minimal typing so rule-based observability can handle bad records later.

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name

In [0]:
STORAGE_ACCOUNT = "hantstorageaccount"
JSON_SOURCE_CONTAINER = "di-json-invoices"
CSV_SOURCE_CONTAINER = "csv-invoices"
LAKEHOUSE_CONTAINER = "lakehouse"

JSON_SOURCE_PATH = f"abfss://{JSON_SOURCE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
CSV_SOURCE_PATH = f"abfss://{CSV_SOURCE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/json_invoice_raw/"
JSON_SCHEMA_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/cloudfiles_schema/json_invoice/"
JSON_CHECKPOINT_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/checkpoints/json_invoice/"

CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/csv_invoice_raw/"
CSV_SCHEMA_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/cloudfiles_schema/csv_invoice/"
CSV_CHECKPOINT_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/checkpoints/csv_invoice/"

In [0]:
csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", CSV_SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(CSV_SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_ts", current_timestamp())
)

(
    csv_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CSV_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(CSV_BRONZE_PATH)
    .awaitTermination()
)

In [0]:
json_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", JSON_SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("multiLine", "true")
    .option("pathGlobFilter", "*.json")
    .load(JSON_SOURCE_PATH)
    .withColumn("_source_file", input_file_name())
    .withColumn("_ingest_ts", current_timestamp())
)

(
    json_stream.writeStream
    .format("delta")
    .option("checkpointLocation", JSON_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(JSON_BRONZE_PATH)
    .awaitTermination()
)

In [0]:
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_BRONZE_CSV_RAW_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_csv_invoice_raw"
BATCH_BRONZE_JSON_RAW_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_json_invoice_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_CSV_RAW_TABLE}
USING DELTA
LOCATION "{CSV_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_JSON_RAW_TABLE}
USING DELTA
LOCATION "{JSON_BRONZE_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_BRONZE_CSV_RAW_TABLE}"))
display(spark.sql(f"DESCRIBE DETAIL {BATCH_BRONZE_JSON_RAW_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0ed73e1e-b2fc-4938-940a-3248669b8015,hant-catalog.invoice.batch_bronze_csv_invoice_raw,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/bronze/csv_invoice_raw,2026-04-23T18:35:14.001Z,2026-04-23T18:35:17Z,List(),List(),1,55515,Map(),1,2,"List(appendOnly, invariants)",Map(),false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a4c52223-fb3a-4b0c-8884-311c64bc0b00,hant-catalog.invoice.batch_bronze_json_invoice_raw,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/bronze/json_invoice_raw,2026-04-23T18:36:00.819Z,2026-04-23T18:36:08Z,List(),List(),36,7878267,Map(),1,2,"List(appendOnly, invariants)",Map(),false


In [0]:
csv_bronze_df = spark.read.format("delta").load(CSV_BRONZE_PATH)
json_bronze_df = spark.read.format("delta").load(JSON_BRONZE_PATH)

print(f"CSV Bronze row count: {csv_bronze_df.count():,}")
print(f"JSON Bronze row count: {json_bronze_df.count():,}")
print("JSON Bronze schema (top level):")
json_bronze_df.printSchema()

CSV Bronze row count: 1,000
JSON Bronze row count: 1,007
JSON Bronze schema (top level):
root
 |-- analyzeResult: string (nullable = true)
 |-- createdDateTime: string (nullable = true)
 |-- lastUpdatedDateTime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)



In [0]:
dbutils.jobs.taskValues.set(key="csv_bronze_path", value=CSV_BRONZE_PATH)
dbutils.jobs.taskValues.set(key="json_bronze_path", value=JSON_BRONZE_PATH)

print("Task values published — bronze_ge can now start.")
print(f"  csv_bronze_path  = {CSV_BRONZE_PATH}")
print(f"  json_bronze_path = {JSON_BRONZE_PATH}")

Task values published — bronze_ge can now start.
  csv_bronze_path  = abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/bronze/csv_invoice_raw/
  json_bronze_path = abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/bronze/json_invoice_raw/
